# Checkpoint V · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_5/03_examen_solutions.ipynb)

Les solutions des deux questions de code (CP5.5, comparée à `nn.RNN` ; CP5.6, l'entraînement avec early stopping), exécutées, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Partie A · Les questions de code

In [2]:
# Data and tools of the code questions CP5.5 and CP5.6 (run this cell first)
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# CP5.5: a mini-batch of 3 sequences of 5 steps with 2 features, and the parameters of a simple RNN of 4 neurons
rng = np.random.default_rng(55)
X_S = rng.normal(size=(3, 5, 2))                       # (N, T, D): batch, time, features
H0_S = np.zeros((3, 4))                                # (N, H): the starting state
WX_S = rng.normal(0.0, 0.8, size=(2, 4))               # (D, H)
WH_S = rng.normal(0.0, 0.8, size=(4, 4))               # (H, H)
B_S = rng.normal(0.0, 0.3, size=4)                     # (H,)


def rnn_cell(x_t, h, Wx, Wh, b):
    """One step of a simple RNN for a mini-batch: x_t (N, D), h (N, H) -> tanh(x_t @ Wx + h @ Wh + b), (N, H)."""
    return np.tanh(x_t @ Wx + h @ Wh + b)


# CP5.6: 3 072 images of Fashion-MNIST in [0, 1], float64 (2 048 to train, 1 024 to validate)
_X6, _Y6 = wb.datasets.load_fashion_mnist("train", n=3072, normalize=True, seed=56)
X_IMG = torch.from_numpy(_X6.astype(np.float64)).unsqueeze(1)      # (3072, 1, 28, 28)
Y_IMG = torch.from_numpy(_Y6.astype(np.int64))                     # (3072,)


def make_cnn(seed=56):
    """A new small CNN with dropout, in float64 (the same starting weights for the same seed)."""
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # (N, 8, 14, 14)
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # (N, 16, 7, 7)
        nn.Flatten(), nn.Dropout(0.25), nn.Linear(16 * 7 * 7, 10)).double()


def make_loaders(seed=56, n_train=2048, n_val=1024):
    """(train_loader, val_loader): mini-batches of 64 shuffled images, then of 300 images (the last one is smaller)."""
    train = TensorDataset(X_IMG[:n_train], Y_IMG[:n_train])
    val = TensorDataset(X_IMG[2048:2048 + n_val], Y_IMG[2048:2048 + n_val])
    return (DataLoader(train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed)),
            DataLoader(val, batch_size=300))


CHECKPOINT = Path(tempfile.gettempdir()) / "cp5_6_best.pt"     # where the try of CP5.6 saves the best weights
_ORIGINALS = {"X_S": X_S.copy(), "H0_S": H0_S.copy()}          # part B restores them if a try changed them
print("X_S:", X_S.shape, "· images:", tuple(X_IMG.shape), "· labels:", tuple(Y_IMG.shape),
      "· CNN parameters:", sum(p.numel() for p in make_cnn().parameters()))

X_S: (3, 5, 2) · images: (3072, 1, 28, 28) · labels: (3072,) · CNN parameters: 9098


### CP5.5 — Écrire `rnn_forward` à partir d'une cellule fournie (notebook, NumPy) 🔨 ★ ⏱️ 10 min · 1,5 point
**Objectif :** dérouler une couche récurrente simple sur toute une séquence, pour un mini-batch.

Avec NumPy seulement. `rnn_cell(x_t, h, Wx, Wh, b)` (fournie) fait un pas : $h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$, avec `x_t` de forme `(N, D)` et `h` de forme `(N, H)`.

a) `rnn_forward(X, h0, Wx, Wh, b)` reçoit `X` de forme `(N, T, D)` (batch, temps, features : la convention `batch_first=True`) et l'état de départ `h0`, de forme `(N, H)`. Elle rend le couple `(H_all, h_last)` : `H_all`, de forme `(N, T, H)`, où `H_all[:, t]` est l'état **après** le pas `t` (`h0` n'y figure pas), et `h_last`, l'état après le dernier pas, de forme `(N, H)`. Elle ne modifie pas `X`. **[1,2]**
b) Avec les données de la cellule (`X_S`, `H0_S`, `WX_S`, `WH_S`, `B_S`), reporte sur ta feuille `H_all[1, 4, 0]` (4 décimales). **[0,3]**

In [3]:
def rnn_forward(X, h0, Wx, Wh, b):
    """(H_all (N, T, H), h_last (N, H)): the states after each step of rnn_cell over X (N, T, D); CP5.5 a)."""
    h, states = h0, []
    for t in range(X.shape[1]):                        # the time axis is axis 1 (batch_first)
        h = rnn_cell(X[:, t], h, Wx, Wh, b)            # the same weights at every step
        states.append(h)
    return np.stack(states, axis=1), h                 # (N, T, H): h0 is not one of the states


H_all_5, h_last_5 = rnn_forward(X_S, H0_S, WX_S, WH_S, B_S)
print("H_all:", H_all_5.shape, "· h_last:", h_last_5.shape, "· H_all[1, 4, 0] =", round(H_all_5[1, 4, 0], 4))
rnn_5 = nn.RNN(2, 4, batch_first=True).double()        # the same layer in PyTorch: W_ih = Wx.T, W_hh = Wh.T
with torch.no_grad():
    rnn_5.weight_ih_l0.copy_(torch.from_numpy(WX_S.T))
    rnn_5.weight_hh_l0.copy_(torch.from_numpy(WH_S.T))
    rnn_5.bias_ih_l0.copy_(torch.from_numpy(B_S))
    rnn_5.bias_hh_l0.zero_()                           # PyTorch has two biases: their sum plays the role of b
    out_5, h_n_5 = rnn_5(torch.from_numpy(X_S), torch.from_numpy(H0_S).unsqueeze(0))
print(f"nn.RNN: largest gap {np.abs(out_5.numpy() - H_all_5).max():.1e} ; h_n = the last output: "
      f"{torch.equal(h_n_5[0], out_5[:, -1])}")

H_all: (3, 5, 4) · h_last: (3, 4) · H_all[1, 4, 0] = -0.4653
nn.RNN: largest gap 4.4e-16 ; h_n = the last output: True


In [4]:
def variant_5(*, recurrent=True, transpose_wh=False, with_h0=False, bias=True):
    """H_all[1, 4, 0] of CP5.5 written another (wrong) way."""
    h, states = H0_S, [H0_S] if with_h0 else []
    for t in range(X_S.shape[1]):
        z = X_S[:, t] @ WX_S + (B_S if bias else 0.0)
        if recurrent:
            z = z + h @ (WH_S.T if transpose_wh else WH_S)
        h = np.tanh(z)
        states.append(h)
    return float(np.stack(states, axis=1)[1, 4, 0])


wb.record("CP5.5b", H_all_5[1, 4, 0], decimals=4, mistakes={
    "l'état précédent n'entre pas dans le calcul : chaque pas reçoit h, l'état rendu par le pas d'avant": variant_5(recurrent=False),
    "H_all contient h0 en première case : la case t est l'état APRÈS le pas t, sans h0": variant_5(with_h0=True),
    "Wh est appliquée transposée : la cellule calcule h @ Wh, avec Wh de forme (H, H)": variant_5(transpose_wh=True),
    "le biais b manque": variant_5(bias=False)})

WB_ANSWER {"decimals": 4, "hash": "1e80b37f8e7deb145c9a22d6cb246ae79ef163ec6040f292ba429bf59ca8045f", "hash_coarse": "aff56759a046a6e16d573e1391c0f6d8ffe5ca4f22c70ce116aaae5c2a184a0d", "id": "CP5.5b", "kind": "float", "magnitude": -1, "mistakes": {"67ff1e93e324f4ad3f4d3d72debbc85f3c4de2fd1e62702c811e0e602a30229a": "le biais b manque", "9840f2335289f19eebb6b54b211aae974ee49fe488b60aa3d5138b2e09a2fc44": "l'état précédent n'entre pas dans le calcul : chaque pas reçoit h, l'état rendu par le pas d'avant", "9bfb3b4b4c5240b1f71f03d0cc0df33a2c477cf99022d1a41bfc21cd596b5f82": "Wh est appliquée transposée : la cellule calcule h @ Wh, avec Wh de forme (H, H)", "e39b8cd5ea068982df3d8fa1f4dda80f7fc5f176b87f6d8306303e22d6206e86": "H_all contient h0 en première case : la case t est l'état APRÈS le pas t, sans h0"}, "sign": -1}
📝 Ex CP5.5b : réponse enregistrée (float, 4 décimale(s)).


### CP5.6 — Entraîner avec early stopping et garder le meilleur modèle (notebook, PyTorch) 🔨 ★★ ⏱️ 20 min · 2 points
**Objectif :** écrire de mémoire une boucle d'entraînement PyTorch complète : modes entraînement et évaluation, loss de validation par exemple, sauvegarde des meilleurs poids, arrêt après `patience` epochs sans progrès.

Avec PyTorch. `make_cnn()` rend un petit CNN neuf (avec dropout, en `float64`) et `make_loaders()` le couple `(train_loader, val_loader)` sur 2 048 et 1 024 images de Fashion-MNIST ; les mini-batches de validation ont 300 images, sauf le dernier (124).

a) `fit_early_stopping(model, train_loader, val_loader, max_epochs, patience, path, lr=1e-3)` : **[1,6]**
- crée **un** optimiseur, `torch.optim.Adam(model.parameters(), lr=lr)` ;
- à chaque epoch (numérotées à partir de 1, au plus `max_epochs`) : en mode entraînement, un passage sur `train_loader` (cross-entropy moyenne du mini-batch sur les logits, `zero_grad`, `backward`, `step`) ; la loss d'entraînement de l'epoch est la moyenne **par exemple** des losses des mini-batches (chacune pesée par son nombre d'exemples) ; puis, en mode évaluation et sans gradient, la loss de validation : la cross-entropy moyenne **par exemple** sur tout `val_loader` ;
- si la loss de validation est **strictement** plus petite que la meilleure vue jusque-là, sauvegarde `model.state_dict()` dans `path` (`torch.save`) et retient l'epoch ; sinon, compte une epoch sans progrès ; l'entraînement s'arrête dès que `patience` epochs **de suite** n'ont pas progressé ;
- à la fin, recharge dans `model` les poids sauvegardés (`torch.load(path, weights_only=True)`) et rend le dictionnaire `{"train_loss": [...], "val_loss": [...], "best_epoch": k}` : une valeur (un `float`) par epoch faite, et l'epoch des poids gardés, numérotée à partir de 1.

b) La cellule d'essai entraîne `make_cnn()` sur `make_loaders()` avec `max_epochs=30`, `patience=3` et `lr=3e-3` (une dizaine de secondes) : reporte sur ta feuille $[\text{best\_epoch}, \text{loss de validation à cette epoch}]$ (4 décimales). **[0,4]**

In [5]:
def fit_early_stopping(model, train_loader, val_loader, max_epochs, patience, path, lr=1e-3):
    """Adam, early stopping on the validation loss (mean per example), best state_dict saved in path and reloaded;
    returns {"train_loss": [...], "val_loss": [...], "best_epoch": k}; CP5.6 a)."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)          # one optimizer: its moments live all along
    history = {"train_loss": [], "val_loss": [], "best_epoch": None}
    best, waited = float("inf"), 0
    for epoch in range(1, max_epochs + 1):
        model.train()                                                # dropout on
        total, count = 0.0, 0
        for X, y in train_loader:
            loss = nn.functional.cross_entropy(model(X), y)          # logits: the softmax is inside
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(y)
            count += len(y)
        history["train_loss"].append(total / count)
        model.eval()                                                 # dropout off
        with torch.no_grad():
            total, count = 0.0, 0
            for X, y in val_loader:
                total += nn.functional.cross_entropy(model(X), y, reduction="sum").item()
                count += len(y)
        val = total / count                                          # per example: the last mini-batch has 124
        history["val_loss"].append(val)
        if val < best:                                               # strictly better: save
            best, waited, history["best_epoch"] = val, 0, epoch
            torch.save(model.state_dict(), path)
        else:
            waited += 1
            if waited == patience:
                break
    model.load_state_dict(torch.load(path, weights_only=True))       # back to the best weights
    return history


history_6 = fit_early_stopping(make_cnn(), *make_loaders(), 30, 3, CHECKPOINT, lr=3e-3)
best_6 = history_6["best_epoch"]
print("epochs done:", len(history_6["val_loss"]), "· best_epoch:", best_6,
      "· its validation loss:", round(history_6["val_loss"][best_6 - 1], 4))
print("validation losses:", [round(v, 4) for v in history_6["val_loss"]])

epochs done: 12 · best_epoch: 9 · its validation loss: 0.4819
validation losses: [0.944, 0.6915, 0.6496, 0.5907, 0.5481, 0.5334, 0.5222, 0.4949, 0.4819, 0.5105, 0.4886, 0.4827]


In [6]:
def variant_6(*, per_batch=False, train_mode=False, extra_wait=0, lr=3e-3, zero_based=False, last=False):
    """[best_epoch, its validation loss] of CP5.6 b) written another (wrong) way."""
    model = make_cnn()
    train_loader, val_loader = make_loaders()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_epoch, waited, val_losses = float("inf"), None, 0, []
    for epoch in range(1, 31):
        model.train()
        for X, y in train_loader:
            loss = nn.functional.cross_entropy(model(X), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if not train_mode:
            model.eval()
        with torch.no_grad():
            losses = [(nn.functional.cross_entropy(model(X), y).item(), len(y)) for X, y in val_loader]
        val = float(np.mean([l for l, _ in losses])) if per_batch else sum(l * n for l, n in losses) / 1024
        val_losses.append(val)
        if val < best:
            best, best_epoch, waited = val, epoch, 0
        else:
            waited += 1
            if waited == 3 + extra_wait:
                break
    return [best_epoch - 1 if zero_based else best_epoch, val_losses[-1] if last else best]


wb.record("CP5.6b", [best_6, history_6["val_loss"][best_6 - 1]], decimals=4, mistakes={
    "la loss de validation est la moyenne des moyennes des mini-batches : le dernier, de 124 images, pèse alors "
    "autant qu'un mini-batch de 300 ; la moyenne est PAR EXEMPLE": variant_6(per_batch=True),
    "la validation tourne en mode entraînement : le dropout éteint des neurones pendant la mesure (model.eval() "
    "avant la validation)": variant_6(train_mode=True),
    "l'arrêt vient une epoch trop tard : il a lieu dès que patience epochs DE SUITE n'ont pas progressé":
    variant_6(extra_wait=1),
    "le learning rate reçu (lr=3e-3) n'est pas celui de l'optimiseur": variant_6(lr=1e-3),
    "les epochs se numérotent à partir de 1": variant_6(zero_based=True),
    "c'est la loss de validation de la dernière epoch faite : on demande celle de l'epoch des poids gardés":
    variant_6(last=True)})

WB_ANSWER {"decimals": 4, "element_coarse": [["d0566571c0c8f37bd0954894d3f61965cfa01f8e33cede512cdd7d5d343c0fbe"], ["59721af34c79ad6f8a7e36b1cdad143f06e0977feacdb62364ed3ea20230c0f9"]], "element_hashes": ["a68b5bb202eb8f428ecf4516425c7997b91534849256fb825d4ae0433975d50a", "d3c3fbd715bb4c1de3b130e7677c004b51faa27f6562347fd9bc6bf6ac151e58"], "hash": "eacc201141d9735c7af4ef9d6f5ed5fd34100cd3bbbca647274e96b33d8d6dd8", "id": "CP5.6b", "kind": "array", "mistakes": {"31f1dbcc12c0ad5d8676f6c968689a15b8321462798d2dd2197f0ed938811f96": "le learning rate reçu (lr=3e-3) n'est pas celui de l'optimiseur", "42fcbc77cec850e5642d62da553c867ad0a1044aa26c9d9a894df37956f1f73d": "la loss de validation est la moyenne des moyennes des mini-batches : le dernier, de 124 images, pèse alors autant qu'un mini-batch de 300 ; la moyenne est PAR EXEMPLE", "6cd3921052ac452a5213aa5d15100776e8062e5a45075472812dd2b5e100773b": "les epochs se numérotent à partir de 1", "7fb12d9ba12dc9d7aa7743a7c8d66ca7b5147a893d0535b226e4

## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [7]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math

import numpy as np

# CP5.2


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


def receptive_field(layers, pool_adds=True):
    """Receptive field of the last layer of [(kernel, stride, is_pool)]: r += (k - 1) * jump, then jump *= stride."""
    r, jump = 1, 1
    for k, s, is_pool in layers:
        if pool_adds or not is_pool:
            r += (k - 1) * jump
        jump *= s
    return r


LAYERS_2 = [(3, 1, False), (2, 2, True), (3, 1, False), (2, 2, True), (3, 2, False)]
PARAMS_2 = [3 * 9 * 16 + 16, 16 * 9 * 32 + 32, 32 * 9 * 64 + 64, 64 * 10 + 10]
SIDE_C_2 = conv_out(8, 3, s=2, p=1)
MACS_2 = 64 * SIDE_C_2 ** 2 * 32 * 9

# CP5.3


def sigmoid(z):
    return 1 / (1 + math.exp(-z))


X_3, C0_3 = 2.0, 0.5
PRE_3 = [0.5 * X_3 - 1, 1.0 * X_3 + 0, 0.25 * X_3 + 0, -0.5 * X_3 + 1]      # i, f, g, o (h0 = 0)
GATES_3 = [sigmoid(PRE_3[0]), sigmoid(PRE_3[1]), math.tanh(PRE_3[2]), sigmoid(PRE_3[3])]
I_3, F_3, G_3, O_3 = GATES_3
C1_3 = F_3 * C0_3 + I_3 * G_3
H1_3 = O_3 * math.tanh(C1_3)

# CP5.8
EMB_8 = 10_000 * 256
LSTM_8 = 4 * (512 * 256 + 512 * 512 + 2 * 512)
OUT_8 = 512 * 10_000 + 10_000
P_8 = EMB_8 + LSTM_8 + OUT_8
FLOP_8 = 6 * P_8 * 1e6

# CP5.10
X_10 = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 9.0])
TRAIN_10 = X_10[2:]                                   # KFold(3) without shuffle: the first fold tests (1, 2)
Z_10 = float((2.0 - TRAIN_10.mean()) / TRAIN_10.std())
XD_10 = np.array([[1.0, 2.0], [0.0, -1.0]])
W_10 = np.array([[1.0, 0.0, -1.0], [2.0, 1.0, 0.0]])
DZ_10 = np.array([[0.5, -1.0, 0.0], [1.0, 0.0, 2.0]])

# CP5.12
X_12, W_12 = np.array([1.0, 2.0, -1.0, 3.0]), np.array([1.0, -1.0])
Y_12 = np.array([W_12 @ X_12[t:t + 2] for t in range(3)])          # cross-correlation, as nn.Conv1d
DW_12 = [float(Y_12 @ X_12[:3]), float(Y_12 @ X_12[1:]), float(Y_12.sum())]
DX_12 = np.zeros(4)
for t in range(3):
    DX_12[t:t + 2] += Y_12[t] * W_12

**CP5.1 — Forme de sortie ou erreur ? Six extraits PyTorch**

In [8]:
wb.record("CP5.1a", "(8, 16, 28, 28)", mistakes={"sans padding, un noyau de taille k tient dans n − k + 1 positions, pas n − k": "(8, 16, 27, 27)",
                     "sans padding, la carte rétrécit : padding vaut 0 par défaut": "(8, 16, 32, 32)",
                     "combien de filtres la couche a-t-elle ? Ce sont les canaux de sortie": "(8, 3, 28, 28)"})
wb.record("CP5.1b", "erreur", mistakes={"PyTorch lit les axes dans l'ordre (N, C, H, W) : quel axe prend-il ici pour les canaux ?": "(8, 16, 32, 32)",
                     "Conv2d(3, …) attend 3 canaux sur l'axe 1 ; combien en trouve-t-il ?": "(8, 16, 32, 3)"})
wb.record("CP5.1c", "(1, 4, 3, 3)", mistakes={"MaxPool2d arrondit la taille vers le bas": "(1, 4, 4, 4)",
                     "le stride de MaxPool2d(2) vaut sa taille, 2 : il divise la taille par 2": "(1, 4, 6, 6)",
                     "un pooling garde les canaux : il agit sur chaque carte séparément": "(1, 2, 3, 3)",
                     "MaxPool2d accepte une taille impaire : il arrondit": "erreur"})
wb.record("CP5.1d", "(2, 7, 20)", mistakes={"sans batch_first, quel axe nn.LSTM prend-il pour le batch ?": "(2, 4, 20)",
                     "[1][0] est h_n, l'état final de chaque couche, pas la sortie à chaque pas": "(4, 7, 20)",
                     "h_n a une ligne par couche empilée": "(1, 7, 20)",
                     "dans h_n, quel axe vient en premier ?": "(7, 2, 20)"})
wb.record("CP5.1e", "(4, 7, 40)", mistakes={"bidirectionnel : la sortie de chaque pas concatène les deux sens": "(4, 7, 20)",
                     "[0] est la sortie de chaque pas, pas h_n": "(2, 4, 20)",
                     "avec batch_first=True, dans quel ordre viennent les axes de la sortie ?": "(7, 4, 40)"})
wb.record("CP5.1f", "(2, 3, 5)", mistakes={"nn.Linear aplatit-il les axes qui précèdent le dernier ?": "(6, 5)",
                     "nn.Linear accepte un nombre quelconque d'axes devant le dernier": "erreur",
                     "que devient le dernier axe dans nn.Linear(16, 5) ?": "(2, 3, 16)"})

WB_ANSWER {"hash": "1a56bc989a4aaa0bebfcd2d099b790475f57ca5181e82aa9653951d570cb4633", "id": "CP5.1a", "kind": "str", "mistakes": {"92d71fc191fb84dd2fe4c1293832f94c8e04189bd3b5379acea8af279b74892f": "sans padding, un noyau de taille k tient dans n − k + 1 positions, pas n − k", "a78370bd532ee9140a3846014e6ef02909b9aae4cbdf2c2ab7e4f3dcb9654132": "sans padding, la carte rétrécit : padding vaut 0 par défaut", "e714ff404a27ce15da6ce4d4029488458c1982758628dd94f4e74091ad194a4d": "combien de filtres la couche a-t-elle ? Ce sont les canaux de sortie"}}
📝 Ex CP5.1a : réponse enregistrée (str).
WB_ANSWER {"hash": "03e4142a6903e8461455bde2feeab0670ae4645d5091e9e713420e704740a0b6", "id": "CP5.1b", "kind": "str", "mistakes": {"340a7c3045afe10136767be92f2f4a98881b612158f4a61c2950936907700026": "Conv2d(3, …) attend 3 canaux sur l'axe 1 ; combien en trouve-t-il ?", "cf526b1e7342df3dd97f4278578a9f073e8e488b256e124e182dab01ccd4424e": "PyTorch lit les axes dans l'ordre (N, C, H, W) : quel axe prend-il ic

**CP5.2 — Formes, paramètres et champ récepteur d'un petit CNN**

In [9]:
wb.record("CP5.2a", "(1, 16, 16, 16)", mistakes={"le bloc A se termine par un MaxPool2d(2)": "(1, 16, 32, 32)",
                     "padding=1 avec un noyau 3 × 3 garde la taille de la carte": "(1, 16, 15, 15)"})
wb.record("CP5.2b", "(1, 32, 8, 8)", mistakes={"le bloc B se termine aussi par un MaxPool2d(2)": "(1, 32, 16, 16)",
                     "combien de filtres la convolution du bloc B a-t-elle ?": "(1, 16, 8, 8)"})
wb.record("CP5.2c", "(1, 64, 4, 4)", mistakes={"cette convolution a un stride de 2 : elle saute une position sur deux": "(1, 64, 8, 8)",
                     "la convolution C a aussi padding=1 : un pixel ajouté de chaque côté de la carte": "(1, 64, 3, 3)",
                     "on demande la sortie de la convolution C, avant AdaptiveAvgPool2d(1)": "(1, 64, 1, 1)"})
wb.record("CP5.2d", [*PARAMS_2, sum(PARAMS_2)], mistakes={"compte aussi les biais, un par filtre ou par neurone": [*(p - b for p, b in zip(PARAMS_2, (16, 32, 64, 10))), sum(PARAMS_2) - 122],
                     "un filtre couvre TOUS les canaux d'entrée": [16 * 9 + 16, 32 * 9 + 32, 64 * 9 + 64, 650, 16 * 10 + 32 * 10 + 64 * 10 + 650],
                     "que reçoit la couche dense, après AdaptiveAvgPool2d(1) ?": [*PARAMS_2[:3], 64 * 16 * 10 + 10, sum(PARAMS_2[:3]) + 64 * 16 * 10 + 10]})
wb.record("CP5.2e", MACS_2, mistakes={"le stride de 2 réduit le nombre de positions de sortie : relis c)": 64 * 8 * 8 * 32 * 9,
                     "chaque position combine les 32 canaux d'entrée, chacun avec 3 × 3 poids": 64 * SIDE_C_2 ** 2 * 9,
                     "on compte une multiplication-addition par poids ET par position de sortie": 32 * 9 * 64})
wb.record("CP5.2f", receptive_field(LAYERS_2), mistakes={"les poolings comptent : chacun agrandit le champ et multiplie l'écart entre positions voisines": receptive_field([(3, 1, False)] * 2 + [(3, 2, False)]),
                     "un pooling 2 × 2 agrandit aussi le champ, de (2 − 1) × l'écart courant": receptive_field(LAYERS_2, pool_adds=False),
                     "l'écart entre deux positions voisines grandit après chaque stride : il multiplie l'apport des couches suivantes": 3 + 1 + 2 + 1 + 2,
                     "le stride d'une couche agrandit le champ des couches SUIVANTES, pas le sien": 26,
                     "chaque couche ajoute (k − 1) × l'écart des positions de son ENTRÉE : l'écart se multiplie par le stride après": 29})

WB_ANSWER {"hash": "9226f3b2d5d32d1e46be0b3ad987ce686fa8f8c808659cd2e7ca248422b5d238", "id": "CP5.2a", "kind": "str", "mistakes": {"60aeca7d543b2d2e470ccee4efb26c368fab9b5a910eb3daf9ea69794f90afe1": "le bloc A se termine par un MaxPool2d(2)", "79a0103e0452334c3ec8b48607e0ac6f08943007fc34c273dc29340904b58d1f": "padding=1 avec un noyau 3 × 3 garde la taille de la carte"}}
📝 Ex CP5.2a : réponse enregistrée (str).
WB_ANSWER {"hash": "b3e5a9660cd3e78d5cf97cd6d08c1af8343e601c5b4d442f8bbf04ef597e611f", "id": "CP5.2b", "kind": "str", "mistakes": {"1939701091793872dbee0be3a7969fb7a8d1bce376a5a45ee6ec9e556bda5477": "combien de filtres la convolution du bloc B a-t-elle ?", "2f0d87bb87a321eb90571449abe8ea181e3fed7d851bfd97e9226e9ef108d111": "le bloc B se termine aussi par un MaxPool2d(2)"}}
📝 Ex CP5.2b : réponse enregistrée (str).
WB_ANSWER {"hash": "de0ffeed0127e1d60a5f0805e95d534bbb2955fa0d7e2c6a03de5c08aa9b3b2f", "id": "CP5.2c", "kind": "str", "mistakes": {"725dccaab8bd50369b5aed4bfaad3cf970e8b

**CP5.3 — Un pas de LSTM à la main**

In [10]:
wb.record("CP5.3a", GATES_3, decimals=4, mistakes={"la proposition g passe par tanh, pas par la sigmoïde": [I_3, F_3, sigmoid(PRE_3[2]), O_3],
                     "les portes i, f et o passent par la sigmoïde, pas par tanh": [math.tanh(PRE_3[0]), math.tanh(PRE_3[1]), G_3, math.tanh(PRE_3[3])],
                     "ce sont les sommes pondérées : applique ensuite la sigmoïde ou tanh": PRE_3})
wb.record("CP5.3b", C1_3, decimals=4, mistakes={"la cellule garde f ⊙ c₀ de son ancien contenu": I_3 * G_3,
                     "la proposition g entre à travers la porte d'entrée : i ⊙ g": F_3 * C0_3 + G_3,
                     "c'est la porte d'oubli qui multiplie l'ancienne cellule, et la porte d'entrée la proposition": I_3 * C0_3 + F_3 * G_3})
wb.record("CP5.3c", H1_3, decimals=4, mistakes={"h = o ⊙ tanh(c) : la cellule passe par tanh": O_3 * C1_3,
                     "la porte de sortie o filtre tanh(c)": math.tanh(C1_3)})
wb.record("CP5.3d", 4 * (20 * 10 + 20 * 20 + 20 + 20), mistakes={"PyTorch a deux vecteurs de biais par porte, b_ih et b_hh": 4 * (20 * 10 + 20 * 20 + 20),
                     "un LSTM a quatre blocs de poids (i, f, g, o), pas un seul": 20 * 10 + 20 * 20 + 20 + 20,
                     "c'est le compte d'un GRU (trois blocs) : le LSTM en a quatre": 3 * (20 * 10 + 20 * 20 + 20 + 20),
                     "relis la forme des poids récurrents : de l'état vers les portes": 4 * (20 * 10 + 20 * 10 + 20 + 20)})

WB_ANSWER {"decimals": 4, "element_coarse": [["cb31690504f9689d82f62f6443a78fa0b3771efc4191586d5c0d901ccb729e81"], ["ccdbecb0dc31d1770459dea7f7717618e04c4e78cb22b36705ba7ea77530dce1"], ["9f0a8b8e9571aa78a5a48f0f4b17d25923b868fcea38f5066a4cca73834f8d98"], ["5c0f8ec26599e2abcbd62c4e5d1eed5f3526f180d96acf55f7eaf4700276473b"]], "element_hashes": ["c337790335e4d5b69a37eb020b4ea502e5b7b4195a01250a8c63ee0f70189f1e", "10a07de6e0427169163e66cc118426c18c83c1b4e6a026ec0508a4abcdd24299", "c973a24b981b0a1ef7f3654f1e91e4915336254ba7586f18aeeb68c2d307b943", "779663163e6cebdda59dc6453b7e3e3097e24f8e1f29c7356b55a7452a6e932f"], "hash": "99f0e42688c4e4afc1e92e6e0d49726589600ab3023dc8837a711a6aea8a2307", "id": "CP5.3a", "kind": "array", "mistakes": {"69baa3e8c1c6e0bb72de358bdfaa578ada7d2ed796c899e35788a52136e1f3a5": "la proposition g passe par tanh, pas par la sigmoïde", "a94d2a441b80a79062b6c26aa5efd035cf46f7a812a97fc8035c2319221567c6": "les portes i, f et o passent par la sigmoïde, pas par tanh", "ad8cc

**CP5.4 — Les lettres des quatre lignes fautives (a)**

In [11]:
wb.record("CP5.4a", "BDFI", mistakes={"il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "DFI",
                     "il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BFI",
                     "il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BDI",
                     "il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDF",
                     "une ligne de trop : la ligne (A) est correcte ; relis ce qu'elle fait": "ABDFI",
                     "une ligne de trop : la ligne (C) est correcte ; relis ce qu'elle fait": "BCDFI",
                     "une ligne de trop : la ligne (E) est correcte ; relis ce qu'elle fait": "BDEFI",
                     "une ligne de trop : la ligne (G) est correcte ; relis ce qu'elle fait": "BDFGI",
                     "une ligne de trop : la ligne (H) est correcte ; relis ce qu'elle fait": "BDFHI",
                     "une ligne de trop : la ligne (J) est correcte ; relis ce qu'elle fait": "BDFIJ",
                     "une ligne de trop : la ligne (K) est correcte ; relis ce qu'elle fait": "BDFIK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "ADFI",
                     "la ligne (C) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "CDFI",
                     "(E) est correcte : l'erreur s'y déclare, mais on demande la ligne à corriger ; quelle forme Conv2d(1, …) attend-elle en entrée ?": "DEFI",
                     "la ligne (G) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "DFGI",
                     "la ligne (H) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "DFHI",
                     "la ligne (J) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "DFIJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : quelle forme Conv2d(1, …) attend-elle en entrée ?": "DFIK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "ABFI",
                     "la ligne (C) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BCFI",
                     "(E) est correcte : l'erreur s'y déclare, mais on demande la ligne à corriger ; refais la taille de la carte après le MaxPool2d(2)": "BEFI",
                     "la ligne (G) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BFGI",
                     "la ligne (H) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BFHI",
                     "la ligne (J) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BFIJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : refais la taille de la carte après le MaxPool2d(2)": "BFIK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "ABDI",
                     "la ligne (C) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BCDI",
                     "la ligne (E) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BDEI",
                     "la ligne (G) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BDGI",
                     "une correction à l'appel (H) peut marcher, si elle transpose l'entrée et prend la sortie du dernier pas (le corrigé l'accepte) ; la correction attendue règle l'ordre des axes une fois pour toutes, là où il se décide": "BDHI",
                     "la ligne (J) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BDIJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : dans quel ordre nn.LSTM lit-il les axes, par défaut ?": "BDIK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "ABDF",
                     "la ligne (C) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BCDF",
                     "la ligne (E) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDEF",
                     "la ligne (G) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDFG",
                     "la ligne (H) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDFH",
                     "la ligne (J) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDFJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?": "BDFK"})

WB_ANSWER {"hash": "094868b9cc06193bf49a09a0c02fc75acecf84d8b778374f2a311089d7fe028e", "id": "CP5.4a", "kind": "str", "mistakes": {"004ee89339545094d1b3c091ceb29529198f9f9e497ed6f2f1f7862047bf04f1": "une correction à l'appel (H) peut marcher, si elle transpose l'entrée et prend la sortie du dernier pas (le corrigé l'accepte) ; la correction attendue règle l'ordre des axes une fois pour toutes, là où il se décide", "03bbe8e853caee19f197694c00cdac2846a8e144729bcd84ba1e5d66139e6363": "une ligne de trop : la ligne (E) est correcte ; relis ce qu'elle fait", "0830732f099ef31bc964034371b03e09c769eaa1cf03a1b2c7df248933f7cd81": "une ligne de trop : la ligne (G) est correcte ; relis ce qu'elle fait", "0ce504e154073e70f800c529a67834444d0f4dd2bd4c3f918fc017d6cea8be80": "la ligne (A) est correcte, et il manque une ligne fautive : quel pas de temps résume toute l'image ?", "0dc6a9aa0b8465f6f9523621f1ca54b0cb500ae6d60955610b61d8c7328aebf8": "la ligne (J) est correcte, et il manque une ligne fautive :

**CP5.7 — Diagnostiquer quatre paires de courbes**

In [12]:
wb.record("CP5.7a", "C", mistakes={"un learning rate trop grand fait sauter ou stagner la loss : ici les deux courbes baissent régulièrement": "A",
                     "l'overfitting met la validation AU-DESSUS de l'entraînement, et la fait remonter : c'est l'inverse ici": "B",
                     "avec des labels au hasard, la loss de validation resterait vers ln 10 ≈ 2,3": "D",
                     "un learning rate trop petit ferait baisser la loss très lentement : elle baisse vite les premières epochs": "E",
                     "eval() oublié laisserait le dropout actif pendant la validation : sa loss monterait au niveau de l'entraînement, pas en dessous": "F"})
wb.record("CP5.7b", "A", mistakes={"l'overfitting suppose une loss d'entraînement qui baisse : celle-ci ne descend jamais sous ln 10 ≈ 2,3": "B",
                     "ce MLP n'a pas de dropout": "C",
                     "des labels au hasard laisseraient la loss d'entraînement baisser lentement, à force de mémoriser : ici elle dépasse 11 à la première epoch, puis reste plate": "D",
                     "un pas trop petit ne fait pas monter la loss au-dessus de 11 à la première epoch": "E",
                     "ce MLP n'a ni dropout ni batchnorm : eval() ne change rien à sa validation": "F"})
wb.record("CP5.7c", "B", mistakes={"la loss d'entraînement baisse régulièrement sous 0,1 : le pas n'est pas trop grand": "A",
                     "ce CNN n'a pas de dropout ; regarde où va la loss de validation après son minimum": "C",
                     "des labels au hasard empêcheraient la validation de descendre sous ln 10 ≈ 2,3": "D",
                     "la loss d'entraînement baisse vite : le pas n'est pas trop petit": "E",
                     "eval() oublié ne fait pas remonter la validation pendant que l'entraînement descend sous 0,1 : cet écart qui se creuse a une autre cause": "F"})
wb.record("CP5.7d", "E", mistakes={"un pas trop grand fait sauter la loss : ici elle baisse, sans un saut": "A",
                     "l'overfitting suppose une loss d'entraînement basse : où en est-elle après 30 epochs ?": "B",
                     "ce MLP n'a pas de dropout": "C",
                     "des labels au hasard ne feraient pas baisser la loss de validation : ici elle baisse avec celle d'entraînement": "D",
                     "ce MLP n'a ni dropout ni batchnorm : eval() ne change rien à sa validation": "F"})

WB_ANSWER {"hash": "b06cdc4ea06c83fad49341ef839fb204735e8460755054e97b31aec80ad1a68e", "id": "CP5.7a", "kind": "str", "mistakes": {"2092c106779f4b2faa788c20932083312402407365d1344fb3ce7ec0c90c0ee7": "avec des labels au hasard, la loss de validation resterait vers ln 10 ≈ 2,3", "256a543249505bb7603e2ff1c10c185d5d92039314ff2f0e3ef1b54b7e4d21cb": "un learning rate trop grand fait sauter ou stagner la loss : ici les deux courbes baissent régulièrement", "b67b681a1ea43d4e58228bfd0c61d0c8e9b9e6c77808306fa97aecebd090c44e": "l'overfitting met la validation AU-DESSUS de l'entraînement, et la fait remonter : c'est l'inverse ici", "e9739ca6332158e1d5e3859ae71823a5cec2413e83be8069da1cf319b29ef921": "un learning rate trop petit ferait baisser la loss très lentement : elle baisse vite les premières epochs", "e9e125c34bd7685028b6b3a3d2315712a6832b3c71fc6933ccd212ade3726861": "eval() oublié laisserait le dropout actif pendant la validation : sa loss monterait au niveau de l'entraînement, pas en dessou

**CP5.8 — Fermi : un générateur de texte LSTM**

In [13]:
wb.record("CP5.8a", P_8 / 1e6, decimals=1, mistakes={"un LSTM a quatre blocs de poids (i, f, g, o)": (EMB_8 + LSTM_8 / 4 + OUT_8) / 1e6,
                     "l'embedding a des paramètres : une ligne par mot du vocabulaire": (LSTM_8 + OUT_8) / 1e6,
                     "n'oublie pas la couche de sortie": (EMB_8 + LSTM_8) / 1e6})
wb.record("CP5.8b", round(100 * OUT_8 / P_8), fractional="arrondis au pourcent près", mistakes={"c'est la part de l'embedding : on demande celle de la couche de sortie": round(100 * EMB_8 / P_8),
                     "c'est la part du LSTM : on demande celle de la couche de sortie": round(100 * LSTM_8 / P_8)})
wb.record("CP5.8c", FLOP_8 / 1e13, decimals=1, mistakes={"la règle compte 6 FLOP par paramètre et par token": P_8 * 1e6 / 1e13,
                     "2PN est le coût de la passe avant seule ; l'entraînement compte aussi la passe arrière, 6PN": 2 * P_8 * 1e6 / 1e13})
wb.record("CP5.8d", round(FLOP_8 / 1e13), fractional="arrondis à la seconde", mistakes={"attention aux puissances de 10 : la machine fait 10¹³ FLOP par seconde": round(FLOP_8 / 1e12)})

WB_ANSWER {"decimals": 1, "hash": "44eabcd01b0affbc029bc66b65b4a2c475aad2aac2add8e7c79e4b9732f04859", "hash_coarse": "4489ad8aa1cff812d7a0c91dcf8f4601d087d4772fcb3c5497b755d42c23ac99", "id": "CP5.8a", "kind": "float", "magnitude": 0, "mistakes": {"4a5811cdd3579e188e5f1d20cf933b650a75152815b691f5565f862cc3d3f3a0": "n'oublie pas la couche de sortie", "c062179dafa1268f5bc9b7ef9056ed68ffce9a11c88e6f2f3f68f322afea1995": "un LSTM a quatre blocs de poids (i, f, g, o)", "d97196e094525803aedcf3063dfc3f4c09780e03665efd0a667dc9225fb48794": "l'embedding a des paramètres : une ligne par mot du vocabulaire"}, "sign": 1}
📝 Ex CP5.8a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"fractional": "arrondis au pourcent près", "hash": "79763e2b0cbf9c4e009e48ef9da8addec32a07261805134b4f97873eba4da71d", "id": "CP5.8b", "kind": "int", "magnitude": 1, "mistakes": {"905da6676f1cabb799f44a179de192a9f5ecb00e294284b117a9f952cf2dc123": "c'est la part de l'embedding : on demande celle de la couche de sorti

**CP5.10 — Retour sur les parties III et IV**

In [14]:
wb.record("CP5.10a", [TRAIN_10.mean(), TRAIN_10.std()], decimals=4, mistakes={"dans un Pipeline, le scaler s'ajuste sur les exemples d'entraînement du fold seulement": [X_10.mean(), X_10.std()],
                     "StandardScaler divise par n, pas par n − 1": [TRAIN_10.mean(), TRAIN_10.std(ddof=1)],
                     "KFold sans shuffle découpe dans l'ordre : le premier fold TESTE les deux premières valeurs et s'entraîne sur les autres": [X_10[:4].mean(), X_10[:4].std()]})
wb.record("CP5.10b", Z_10, decimals=3, mistakes={"dans un Pipeline, le scaler s'ajuste sur les exemples d'entraînement du fold seulement": float((2.0 - X_10.mean()) / X_10.std()),
                     "StandardScaler divise par n, pas par n − 1": float((2.0 - TRAIN_10.mean()) / TRAIN_10.std(ddof=1)),
                     "KFold sans shuffle découpe dans l'ordre : le premier fold TESTE les deux premières valeurs et s'entraîne sur les autres": float((2.0 - X_10[:4].mean()) / X_10[:4].std())})
wb.record("CP5.10c", 4 * 3 + 1, mistakes={"refit=True (le défaut) réentraîne ensuite le meilleur pipeline sur tout le dataset, scaler compris": 12,
                     "le Pipeline réajuste le scaler dans chaque fold, pour chaque valeur de C": 1,
                     "chaque valeur de C est évaluée sur 3 folds, et chaque fold ajuste son propre scaler": 4,
                     "il y a 4 valeurs de C à évaluer, pas une seule": 3,
                     "on demande les ajustements de ce GridSearchCV seulement, sans ceux du cross_val_score d'avant": 16})
wb.record("CP5.10d", (XD_10.T @ DZ_10).tolist(), decimals=1, mistakes={"relis la règle : ∂L/∂W doit avoir la forme de W": (XD_10 @ DZ_10).tolist()})
wb.record("CP5.10e", DZ_10.sum(axis=0).tolist(), decimals=1, mistakes={"∂L/∂b SOMME les deltas sur les exemples : la moyenne est déjà dans ∂L/∂Z": DZ_10.mean(axis=0).tolist(),
                     "on somme sur les exemples (les lignes), pour garder un gradient par neurone": DZ_10.sum(axis=1).tolist()})
wb.record("CP5.10f", (DZ_10 @ W_10.T).tolist(), decimals=1, mistakes={"∂L/∂X a la forme de X, (exemples, features) : une ligne par exemple": (DZ_10 @ W_10.T).T.tolist()})

WB_ANSWER {"decimals": 4, "element_coarse": [["3f261a698e960c3bccee2a0cb2c7359bcce75b960a49df829b0217cdf6acd1b0"], ["1047693327b499d84da154a42a8773b064d5a298b4e0bb85b1b74edf8eb91a4e"]], "element_hashes": ["39318c04f0ce6b9223c7a7ab8351c44ca38cf53f4452d83e71f77ef9bd22a437", "f21a482da58ec21f93750d75d91c2da4ac2427125e53225bc72e513b9bcfcea0"], "hash": "5c4756d5414f7e65f40aee3592ebdc7a0c5c6b5ea0c16fe2ce3c1e12cfe1ebf5", "id": "CP5.10a", "kind": "array", "mistakes": {"143c48b5734c97c33ad5702c4fb3bf3aaef9c9159140c21bcf3bf70891da9404": "StandardScaler divise par n, pas par n − 1", "73a71e45a0f8a4845b994f69fb4674e78a45fcc98fd0e310509294a276c66cb1": "dans un Pipeline, le scaler s'ajuste sur les exemples d'entraînement du fold seulement", "b5d1fc63d2fd702cfcf32ee715e828dae2c836d139a1bc891413ac907551d84c": "KFold sans shuffle découpe dans l'ordre : le premier fold TESTE les deux premières valeurs et s'entraîne sur les autres"}, "shape": [2]}
📝 Ex CP5.10a : réponse enregistrée (array, 4 décimale(s))

**CP5.11 — Questions flash : paramètres et tailles**

In [15]:
wb.record("CP5.11a", 8 * 4 * 5 + 8, mistakes={"un filtre 1D couvre tous les canaux d'entrée": 8 * 5 + 8,
                     "compte aussi les biais, un par filtre": 8 * 4 * 5})
wb.record("CP5.11b", 3 * (20 * 10 + 20 * 20 + 20 + 20), mistakes={"PyTorch a deux vecteurs de biais par bloc, b_ih et b_hh": 3 * (20 * 10 + 20 * 20 + 20),
                     "c'est le compte d'un LSTM (quatre blocs) : le GRU en a trois": 4 * (20 * 10 + 20 * 20 + 20 + 20)})
wb.record("CP5.11c", (16 - 1) * 2 - 2 * 1 + 4, mistakes={"c'est la formule d'une convolution ordinaire : la convolution transposée agrandit": conv_out(16, 4, s=2, p=1),
                     "le padding d'une convolution transposée retire des lignes à sa sortie": (16 - 1) * 2 + 4})
wb.record("CP5.11d", 5000 * 64, mistakes={"un embedding n'a pas de biais : une ligne de 64 valeurs par token": 5000 * 64 + 64})

WB_ANSWER {"hash": "9bfef49ccdd5bf3bfe15c4724d94edcc154fd8aac8612d578f8d9b17267fbc11", "id": "CP5.11a", "kind": "int", "magnitude": 2, "mistakes": {"17bf0d60e8e6a2b983b87ec64092af19f346e60988a8ac3c1d19582d11a84eda": "un filtre 1D couvre tous les canaux d'entrée", "f1c7c91504ffc84e554c72d6216c7af7f55bb312dde729d2e655b307e4f292e1": "compte aussi les biais, un par filtre"}, "sign": 1}
📝 Ex CP5.11a : réponse enregistrée (int).
WB_ANSWER {"hash": "ad72473d5da7d6cde417527c5e014f8d329fd2f693afa547a9f0fa60e544ae5a", "id": "CP5.11b", "kind": "int", "magnitude": 3, "mistakes": {"248f952446a14a7a45adb2f1da3b7f9fff3b619e6f5af9e0c61ac2b454a92d77": "PyTorch a deux vecteurs de biais par bloc, b_ih et b_hh", "d2c5a2df1561154b4a0081c9b903a2536247840f541af12513511845b4326760": "c'est le compte d'un LSTM (quatre blocs) : le GRU en a trois"}, "sign": 1}
📝 Ex CP5.11b : réponse enregistrée (int).
WB_ANSWER {"hash": "d4da083e5eee8f8290680605358e267fdedcdf5d177d125bb6ec10c6da6e5bd0", "id": "CP5.11c", "kind": 

**CP5.12 — Rétropropager dans une convolution 1D**

In [16]:
wb.record("CP5.12a", Y_12.tolist(), mistakes={"nn.Conv1d (et le ch. 21) calcule une corrélation croisée : le noyau n'est pas retourné": [-v for v in Y_12.tolist()]})
wb.record("CP5.12c", DW_12, decimals=2, mistakes={"la loss de l'énoncé est une SOMME sur les trois positions, pas une moyenne": [v / 3 for v in DW_12],
                     "w₁ multiplie x_t et w₂ multiplie x_{t+1} : chaque poids voit sa propre fenêtre d'entrées": [DW_12[1], DW_12[0], DW_12[2]]})
wb.record("CP5.12d", DX_12.tolist(), mistakes={"x₂ et x₃ servent à deux positions : leurs gradients additionnent deux contributions": [Y_12[0] * W_12[0], Y_12[1] * W_12[0], Y_12[2] * W_12[0], Y_12[2] * W_12[1]]})

WB_ANSWER {"decimals": 0, "element_hashes": ["2ff39378a2e067d9d6d2d7603630da05038c6746920b3e121f51e2b8dfcac701", "59b1e7899dd957d94678531b4ddef0bf10f7e6c45ea7317212586cbcf76d4b1a", "153df715a0bd60bbb23422d201af6e7c4d315d32d46dccc8ba7e517d14affb4b"], "hash": "67aefdec1d49d4bf2c87fe66adc8d22409303d370dcda2980a75d248deafa985", "id": "CP5.12a", "integer": true, "kind": "array", "mistakes": {"b5fe867758d5d8093693cbe3ac9123bacd2342356766b588125a462fd46849f0": "nn.Conv1d (et le ch. 21) calcule une corrélation croisée : le noyau n'est pas retourné"}, "shape": [3]}
📝 Ex CP5.12a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 2, "element_coarse": [["b96e2807d2a7f9d32a9c832568d4f658bd60e7c06b7419d4af3e17273264c246"], ["d22cfb6638041fd41734481ffdc99af1c596688b5b50fa5ca26cb096b5ca2c43"], ["c1f71bdd0bbc2a8bdc3bbb1463a3e0fe6f3f62df5a0a2d10183395250d08ae2a"]], "element_hashes": ["7ccafc67704a9fd4f2040ae4dbf2230add44ca882ce5a782afb3e1ec7a5ac56c", "172438c8dd4561d60c16c4141851a6c9cf